# Test AI4BI on SageMaker

Upload this whole `SageMaker` folder to a SageMaker Studio notebook or notebook instance, then run the cells below. No Dataiku connection and no external model are required. The sample datasets travel with the folder.

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "settings.yaml").exists():
    ROOT = ROOT.parent
os.environ["AIBI_PROJECT_ROOT"] = str(ROOT)
os.environ["AIBI_LLM_MODE"] = "local"
sys.path[:0] = [str(ROOT / "lib" / "python"), str(ROOT)]
print(ROOT)

In [ ]:
import subprocess, sys
from pathlib import Path
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(Path(os.environ["AIBI_PROJECT_ROOT"]) / "requirements.txt")])

In [ ]:
from sagemaker_app.handler import invoke
import json

result = invoke({
    "question": "What is total Agg Productivity?",
    "domain": "generic_cube",
})
print(json.dumps({
    "status": result["status"],
    "answer": result["answer"],
    "sql": result["sql_query"],
    "table": result["table"],
}, indent=2))

To call a deployed endpoint instead of the in-notebook handler, set `ENDPOINT_NAME` and run the next cell.

In [ ]:
import os, json, boto3

endpoint = os.environ.get("ENDPOINT_NAME", "")
if not endpoint:
    print("ENDPOINT_NAME is not set. The in-notebook test above is the local SageMaker check.")
else:
    runtime = boto3.client("sagemaker-runtime")
    response = runtime.invoke_endpoint(
        EndpointName=endpoint,
        ContentType="application/json",
        Body=json.dumps({"question": "What is total Agg Productivity?", "domain": "generic_cube"}).encode(),
    )
    print(response["Body"].read().decode())